# Flight Satisfaction — CatBoost K-Fold Ensemble

Clean, self-contained pipeline:
1. **Load** `train.csv` / `test.csv`
2. **Median imputation** via `SimpleImputer` fitted on train only
3. **Outlier capping** — 99th-percentile on delay columns
4. **Feature engineering** — delay aggregates, service composite scores, interaction features
5. **5-fold Stratified K-Fold CatBoost ensemble** (`iterations=3000`, `lr=0.05`, `depth=8`, `eval_metric=AUC`)
   — raw string categoricals passed directly to CatBoost, no label-encoding
6. **Save** averaged test probabilities to `result_with_cat_boost.csv`

## 1. Load Data

In [1]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier

train = pd.read_csv('train.csv')
test  = pd.read_csv('test.csv')

print(f'Train: {train.shape}  |  Test: {test.shape}')
print(f'Missing values in train:\n{train.isnull().sum()[train.isnull().sum() > 0]}')
print(f'Missing values in test:\n{test.isnull().sum()[test.isnull().sum() > 0]}')

Train: (699635, 23)  |  Test: (299844, 22)
Missing values in train:
Arrival Delay in Minutes    292
dtype: int64
Missing values in test:
Arrival Delay in Minutes    130
dtype: int64


## 2. Preprocessing

- Drop `id`; save test IDs separately for the submission
- `SimpleImputer(strategy='median')` fitted on train numerics, applied to both train and test
- Cap delay columns at their 99th-percentile training value

In [2]:
# ── Separate IDs and target ───────────────────────────────────────────────────
test_ids = test['id'].copy()

df_train = train.drop(columns=['id']).copy()
df_test  = test.drop(columns=['id']).copy()

# Encode target bool → int
df_train['satisfaction'] = df_train['satisfaction'].astype(int)

# ── Median imputation on all numeric columns ──────────────────────────────────
cat_cols = ['Gender', 'Customer Type', 'Type of Travel', 'Class']
numeric_cols = [c for c in df_train.columns
                if c not in cat_cols and c != 'satisfaction']

imputer = SimpleImputer(strategy='median')
df_train[numeric_cols] = imputer.fit_transform(df_train[numeric_cols])
df_test[numeric_cols]  = imputer.transform(df_test[numeric_cols])

print(f'Numeric columns imputed ({len(numeric_cols)}): {numeric_cols}')
print(f'Missing after imputation — train: {df_train[numeric_cols].isnull().sum().sum()}  '
      f'test: {df_test[numeric_cols].isnull().sum().sum()}')

# ── Cap delay outliers at 99th-percentile thresholds (computed on train) ──────
delay_cols = ['Departure Delay in Minutes', 'Arrival Delay in Minutes']
caps = {}
for col in delay_cols:
    cap = df_train[col].quantile(0.99)
    caps[col] = cap
    df_train[col] = df_train[col].clip(upper=cap)
    df_test[col]  = df_test[col].clip(upper=cap)

print('99th-percentile caps applied:')
for col, cap in caps.items():
    print(f'  {col}: {cap:.1f} min')

# ── Ensure cat cols are clean strings (CatBoost requires no NaN in cat cols) ──
for col in cat_cols:
    df_train[col] = df_train[col].fillna('__MISSING__').astype(str)
    df_test[col]  = df_test[col].fillna('__MISSING__').astype(str)

Numeric columns imputed (17): ['Age', 'Flight Distance', 'Inflight wifi service', 'Departure/Arrival time convenient', 'Ease of Online booking', 'Gate location', 'Food and drink', 'Online boarding', 'Seat comfort', 'Inflight entertainment', 'On-board service', 'Leg room service', 'Baggage handling', 'Checkin service', 'Cleanliness', 'Departure Delay in Minutes', 'Arrival Delay in Minutes']
Missing after imputation — train: 0  test: 0
99th-percentile caps applied:
  Departure Delay in Minutes: 30.0 min
  Arrival Delay in Minutes: 28.0 min


## 3. Feature Engineering

All features computed from existing columns — same formula applied to train and test.

| Group | Features |
|---|---|
| Delay | `total_delay`, `delay_diff`, `delay_recovered`, `any_delay` |
| Service composites | `comfort_score`, `digital_score`, `inflight_exp_score`, `ground_score` |
| Interactions | `service_range`, `avg_service_rating`, `dist_per_age` |

In [3]:
def add_features(df):
    """Apply all feature engineering steps in-place and return the dataframe."""
    dep = df['Departure Delay in Minutes']
    arr = df['Arrival Delay in Minutes']

    df['total_delay']      = dep + arr
    df['delay_diff']       = arr - dep
    df['delay_recovered']  = (df['delay_diff'] < 0).astype(int)
    df['any_delay']        = ((dep > 0) | (arr > 0)).astype(int)

    comfort_cols  = ['Seat comfort', 'Leg room service']
    digital_cols  = ['Inflight wifi service', 'Online boarding', 'Ease of Online booking']
    inflight_cols = ['Inflight entertainment', 'Food and drink', 'Cleanliness', 'On-board service']
    ground_cols   = ['Gate location', 'Checkin service', 'Baggage handling']
    all_svc_cols  = comfort_cols + digital_cols + inflight_cols + ground_cols + \
                    ['Departure/Arrival time convenient']

    df['comfort_score']      = df[comfort_cols].mean(axis=1)
    df['digital_score']      = df[digital_cols].mean(axis=1)
    df['inflight_exp_score'] = df[inflight_cols].mean(axis=1)
    df['ground_score']       = df[ground_cols].mean(axis=1)
    df['service_range']      = df[all_svc_cols].max(axis=1) - df[all_svc_cols].min(axis=1)
    df['avg_service_rating'] = df[all_svc_cols].mean(axis=1)
    df['dist_per_age']       = df['Flight Distance'] / df['Age']

    return df

df_train = add_features(df_train)
df_test  = add_features(df_test)

print(f'Train shape after feature engineering: {df_train.shape}')
print(f'Test  shape after feature engineering: {df_test.shape}')

Train shape after feature engineering: (699635, 33)
Test  shape after feature engineering: (299844, 32)


## 4. Prepare X / y

In [4]:
X = df_train.drop(columns=['satisfaction'])
y = df_train['satisfaction']

X_test_final = df_test[X.columns]   # align column order

print(f'X: {X.shape}  |  X_test_final: {X_test_final.shape}')
print(f'Class balance: {y.value_counts(normalize=True).round(3).to_dict()}')

X: (699635, 32)  |  X_test_final: (299844, 32)
Class balance: {0: 0.556, 1: 0.444}


## 5. CatBoost 5-Fold K-Fold Ensemble

- **`StratifiedKFold`** preserves class balance in every fold
- **10 % internal hold-out** carved from each fold's training slice for early stopping — the validation fold is never touched during training (no leakage)
- Each fold's model predicts on the full test set; 5 probability arrays are **averaged**
- OOF probabilities give a clean overall AUC estimate on the training set

In [5]:
N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

test_probas = np.zeros(len(X_test_final))
oof_probas  = np.zeros(len(X))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), start=1):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx],  y.iloc[val_idx]

    # 10 % internal hold-out for early stopping — validation fold stays clean
    n_es = int(len(X_tr) * 0.1)
    X_es, y_es = X_tr.iloc[-n_es:], y_tr.iloc[-n_es:]
    X_tr, y_tr = X_tr.iloc[:-n_es],  y_tr.iloc[:-n_es]

    model_fold = CatBoostClassifier(
        iterations=3000,
        cat_features=cat_cols,
        learning_rate=0.05,
        depth=8,
        loss_function='Logloss',
        eval_metric='AUC',
        random_seed=42,
        verbose=200,
        allow_writing_files=False,
        thread_count=-1,
    )
    model_fold.fit(X_tr, y_tr, eval_set=(X_es, y_es), early_stopping_rounds=150)

    oof_probas[val_idx]  = model_fold.predict_proba(X_val)[:, 1]
    test_probas         += model_fold.predict_proba(X_test_final)[:, 1]

    fold_auc = roc_auc_score(y_val, oof_probas[val_idx])
    print(f'Fold {fold}/{N_SPLITS}  |  best iter: {model_fold.best_iteration_:>4}  |  OOF AUC: {fold_auc:.5f}')

test_probas /= N_SPLITS

overall_oof_auc = roc_auc_score(y, oof_probas)
print(f'\nOverall OOF AUC: {overall_oof_auc:.5f}')

0:	test: 0.9432491	best: 0.9432491 (0)	total: 355ms	remaining: 17m 44s
200:	test: 0.9578787	best: 0.9578787 (200)	total: 37.1s	remaining: 8m 36s
400:	test: 0.9585147	best: 0.9585151 (394)	total: 1m 13s	remaining: 7m 54s
600:	test: 0.9586789	best: 0.9586840 (599)	total: 1m 50s	remaining: 7m 19s
800:	test: 0.9588255	best: 0.9588277 (799)	total: 2m 27s	remaining: 6m 44s
1000:	test: 0.9588729	best: 0.9588861 (977)	total: 3m 4s	remaining: 6m 7s
1200:	test: 0.9589227	best: 0.9589485 (1138)	total: 3m 41s	remaining: 5m 31s
1400:	test: 0.9589862	best: 0.9590067 (1386)	total: 4m 17s	remaining: 4m 54s
Stopped by overfitting detector  (150 iterations wait)

bestTest = 0.9590066811
bestIteration = 1386

Shrink model to first 1387 iterations.
Fold 1/5  |  best iter: 1386  |  OOF AUC: 0.95849
0:	test: 0.9430465	best: 0.9430465 (0)	total: 186ms	remaining: 9m 18s
200:	test: 0.9579220	best: 0.9579220 (200)	total: 37.8s	remaining: 8m 46s
400:	test: 0.9585532	best: 0.9585532 (400)	total: 1m 16s	remaining:

## 6. Save Submission

In [6]:
result_with_cat_boost = pd.DataFrame({
    'id'          : test_ids.values,
    'satisfaction': test_probas,
})

result_with_cat_boost.to_csv('result_with_cat_boost.csv', index=False)

print(f'Saved result_with_cat_boost.csv  ({len(result_with_cat_boost):,} rows)')
display(result_with_cat_boost.head(10))

Saved result_with_cat_boost.csv  (299,844 rows)


,id,satisfaction
0,699635,0.263078
1,699636,0.907845
2,699637,0.895290
3,699638,0.064838
4,699639,0.060859
5,699640,0.962231
6,699641,0.024783
7,699642,0.928448
8,699643,0.979416
9,699644,0.027736
